# Classificazione del fondo stradale con il dataset PVS

**Autori:** Dennis Brancaleoni, Anxhelo Alla

**Corso:** Machine Learning, 3° anno

## Indice

- 1. Introduzione e descrizione del dataset
- 2. Caricamento ed esplorazione dei dati
- 3. Segmentazione in finestre ed estrazione delle feature
- 4. Preprocessing: valori mancanti, standardizzazione, codifica, selezione feature
- 5. Strategia di validazione e baseline
- 6. Modelli e tuning degli iperparametri
- 7. Ensemble: Bagging e Voting
- 8. Valutazione sul test esterno
- 9. Analisi aggiuntive: posizione dei sensori, effetto della velocità
- 10. Explainability: SHAP e UMAP
- 11. Discussione e lavori futuri

In [8]:
# add the repository root to the path
# so that the src/ package can be imported without installing it.
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config

print("Project root:", config.ROOT)
print("RANDOM_STATE:", config.RANDOM_STATE)
print("Sampling rate:", config.SAMPLING_RATE_HZ, "Hz")

Project root: /Volumes/BRANCA-SSD/UNI/3ANNO/PRIMO_SEMESTRE/MACHINE_LEARNING/ml-from-scratch
RANDOM_STATE: 42
Sampling rate: 100 Hz


## 1. Introduzione e descrizione del dataset

**Dataset.** PVS – Passive Vehicular Sensors (Kaggle, "PVS - Passive Vehicular Sensors Datasets"): 9 registrazioni di guida reale, circa 3 ore in totale, ottenute con 3 veicoli × 3 scenari (percorsi). Ogni veicolo è guidato da un proprio guidatore.

| Registrazioni | Veicolo | Guidatore | Scenari (1, 2, 3) |
|---|---|---|---|
| PVS1–3 | Volkswagen Saveiro | 1 | 13,8 / 11,6 / 10,7 km |
| PVS4–6 | Fiat Bravo | 2 | 13,8 / 11,6 / 10,7 km |
| PVS7–9 | Fiat Palio | 3 | 13,8 / 11,6 / 10,7 km |

**Sensori.** Moduli MPU-9250 a 100 Hz (accelerometro in m/s², giroscopio in °/s, magnetometro in µT, temperatura del sensore in °C) su ciascun lato dell'asse anteriore: sul braccio della sospensione (*sotto* la sospensione), sulla carrozzeria sopra la ruota (*sopra* la sospensione) e sul cruscotto. Il GPS (1 Hz) fornisce la velocità in m/s.

**Target.** Tipo di fondo stradale, classificazione a 3 classi: asfalto, pavé, sterrato (`asphalt_road`, `cobblestone_road`, `dirt_road`).

**Limiti noti.** Ogni auto ha un solo guidatore (effetto auto ed effetto guidatore non separabili); le tre auto percorrono gli stessi tre percorsi; la velocità è un possibile confondente.

## 2. Caricamento ed esplorazione dei dati

In [9]:
# setup: imports, constants and autoreload (so edits to src/ are picked up)
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (CAR_NAMES, DATA_RAW, FIGURES, RECORDING_TO_CAR,
                        RECORDING_TO_SCENARIO, SAMPLING_RATE_HZ)
from src.data_loading import SURFACE_NAMES, load_all, load_recording, sensor_columns

pd.set_option("display.max_columns", 40)

SURFACE_COLS = list(SURFACE_NAMES)        # raw label columns of the three surfaces
RECORDINGS = range(1, 10)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Esplorazione di PVS1

In [10]:
# files of one recording
rec_dir = DATA_RAW / "PVS1"
for p in sorted(rec_dir.iterdir()):
    print(f"{p.name:35s} {p.stat().st_size / 1e6:8.1f} MB")

.claude-flow                             0.0 MB
dataset_gps.csv                          0.2 MB
dataset_gps_mpu_left.csv                84.5 MB
dataset_gps_mpu_right.csv               84.7 MB
dataset_labels.csv                       4.2 MB
dataset_mpu_left.csv                    75.6 MB
dataset_mpu_right.csv                   75.8 MB
dataset_settings_left.csv                0.0 MB
dataset_settings_right.csv               0.0 MB


In [11]:
# load the three tables of one recording (left side)
sensors = pd.read_csv(rec_dir / "dataset_gps_mpu_left.csv")
labels = pd.read_csv(rec_dir / "dataset_labels.csv")
gps = pd.read_csv(rec_dir / "dataset_gps.csv")

print("sensors:", sensors.shape, "| labels:", labels.shape, "| gps:", gps.shape)
display(sensors.head(3))
sensors.info()

sensors: (144036, 32) | labels: (144036, 14) | gps: (1467, 20)


,timestamp,acc_x_dashboard,acc_y_dashboard,acc_z_dashboard,acc_x_above_suspension,acc_y_above_suspension,acc_z_above_suspension,acc_x_below_suspension,acc_y_below_suspension,acc_z_below_suspension,gyro_x_dashboard,gyro_y_dashboard,gyro_z_dashboard,gyro_x_above_suspension,gyro_y_above_suspension,gyro_z_above_suspension,gyro_x_below_suspension,gyro_y_below_suspension,gyro_z_below_suspension,mag_x_dashboard,mag_y_dashboard,mag_z_dashboard,mag_x_above_suspension,mag_y_above_suspension,mag_z_above_suspension,temp_dashboard,temp_above_suspension,temp_below_suspension,timestamp_gps,latitude,longitude,speed
0,1.577219e+09,0.365116,0.167893,9.793961,0.327626,0.172733,9.781861,0.024797,0.172611,9.793824,-0.133896,-0.018883,0.138092,0.138446,0.159659,-0.072572,-0.041780,0.167302,-0.078110,22.878922,-6.040509,1.767874,75.340967,-7.537556,-0.334800,35.065355,33.340132,31.782640,1.577219e+09,-27.717841,-51.098865,0.009128
1,1.577219e+09,0.392649,0.176273,9.771216,0.381496,0.189492,9.699261,0.024797,0.194158,9.842905,-0.027084,-0.003624,0.000763,0.168963,0.068106,0.095274,0.019255,0.304631,0.150771,22.878922,-5.854647,2.089306,75.300243,-6.469241,-5.222873,34.993471,33.172402,31.782640,1.577219e+09,-27.717841,-51.098865,0.009128
2,1.577219e+09,0.409408,0.181062,9.732909,0.283333,0.182310,9.807000,0.003249,0.227677,9.888395,0.125504,-0.186729,-0.090790,-0.136213,0.159659,0.156310,-0.377473,-0.122615,0.028701,23.434686,-4.181891,0.642863,75.463140,-7.893661,-4.687194,34.993471,33.412017,31.926408,1.577219e+09,-27.717841,-51.098865,0.009128


<class 'pandas.DataFrame'>
RangeIndex: 144036 entries, 0 to 144035
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   timestamp                144036 non-null  float64
 1   acc_x_dashboard          144036 non-null  float64
 2   acc_y_dashboard          144036 non-null  float64
 3   acc_z_dashboard          144036 non-null  float64
 4   acc_x_above_suspension   144036 non-null  float64
 5   acc_y_above_suspension   144036 non-null  float64
 6   acc_z_above_suspension   144036 non-null  float64
 7   acc_x_below_suspension   144036 non-null  float64
 8   acc_y_below_suspension   144036 non-null  float64
 9   acc_z_below_suspension   144036 non-null  float64
 10  gyro_x_dashboard         144036 non-null  float64
 11  gyro_y_dashboard         144036 non-null  float64
 12  gyro_z_dashboard         144036 non-null  float64
 13  gyro_x_above_suspension  144036 non-null  float64
 14  gyro_y_above_su

In [12]:
# column groups
sensor_cols = sensor_columns(sensors)
other_cols = [c for c in sensors.columns if c not in sensor_cols]
print(len(sensor_cols), "sensor columns, e.g.", sensor_cols[:3])
print("other columns:", other_cols)

27 sensor columns, e.g. ['acc_x_dashboard', 'acc_y_dashboard', 'acc_z_dashboard']
other columns: ['timestamp', 'timestamp_gps', 'latitude', 'longitude', 'speed']


In [13]:
# labels haven no timestamp: they can only be aligned by row position
assert len(sensors) == len(labels), "sensors and labels have different lengths"

step = sensors["timestamp"].diff().dropna()
print(f"sampling step: median {step.median():.4f},s, max {step.max():.4f}s")
duration_min = (sensors["timestamp"].iloc[-1] - sensors["timestamp"].iloc[0]) / 60
print(f"duration: {duration_min:.1f} min")

sampling step: median 0.0100,s, max 0.0100s
duration: 24.0 min


In [14]:
# label columns: samples per label, and consistency between related labels
print(labels.sum())
print(labels[SURFACE_COLS].sum(axis=1).value_counts())     # expect only 1
print("paved == cobblestone + asphalt:",
      (labels["paved_road"] == labels["cobblestone_road"] + labels["asphalt_road"]).all())
has_bump = labels[["speed_bump_asphalt", "speed_bump_cobblestone"]].sum(axis=1) > 0
print("no_speed_bump == 0 matches the bump columns:",
      ((labels["no_speed_bump"] == 0) == has_bump).all())

paved_road                118168
unpaved_road               25868
dirt_road                  25868
cobblestone_road           61659
asphalt_road               56509
no_speed_bump             140581
speed_bump_asphalt           626
speed_bump_cobblestone      2829
good_road_left             56577
regular_road_left          65304
bad_road_left              22155
good_road_right            56595
regular_road_right         66038
bad_road_right             21403
dtype: int64
1    144036
Name: count, dtype: int64
paved == cobblestone + asphalt: True
no_speed_bump == 0 matches the bump columns: True


In [15]:
# speed: GPS speed is in m/s
# check sample-and-hold vs interpolation
print(sensors["speed"].describe())
print("max speed: %.1f km/h" % (sensors["speed"].max() * 3.6))

# 1 Hz GPS vs 100 Hz MPU: held constant -> speed changes in ~1% of samples
print("fraction of samples where speed changes:", (sensors["speed"].diff() != 0).mean())

count    144036.000000
mean          9.556871
std           7.746386
min           0.002526
25%           4.508887
50%           6.618945
75%          16.647470
max          26.874480
Name: speed, dtype: float64
max speed: 96.7 km/h
fraction of samples where speed changes: 0.009664250604015663


**Osservazioni.** I file `dataset_gps_mpu_*` contengono già i campioni MPU a 100 Hz con la velocità GPS associata (in m/s): non serve fare noi il merge con `dataset_gps.csv`. La velocità resta costante tra due misure GPS (cambia nello 0,97% dei campioni, circa 1 su 100), quindi la velocità media di una finestra è un'approssimazione a gradini. Le etichette non hanno timestamp: hanno lo stesso numero di righe dei sensori e le allineiamo per posizione. Le classi sono coerenti tra loro (`paved = cobblestone + asphalt`, `unpaved = dirt`). Il target a 3 classi è definito da `dirt_road`, `cobblestone_road`, `asphalt_road`.

## 3. Segmentazione in finestre ed estrazione delle feature

## 4. Preprocessing: valori mancanti, standardizzazione, codifica, selezione feature

## 5. Strategia di validazione e baseline

## 6. Modelli e tuning degli iperparametri

## 7. Ensemble: Bagging e Voting

## 8. Valutazione sul test esterno

## 9. Analisi aggiuntive: posizione dei sensori, effetto della velocità

## 10. Explainability: SHAP e UMAP

## 11. Discussione e lavori futuri